## 0. 프로젝트 소개
### 0-1. T우주 외부 서비스 제공 영역 소개
T우주 외부 제공 서비스는 크게 상품 및 계약의 상태를 조회하는 **1) 채널 연동 서비스**와 실제 연동형 상품을 제공하기 위해 제휴사와 가입/해지/갱신 등을 처리하는 **2) 제휴사 연동 서비스**로 구성

<center>
<img src="https://raw.githubusercontent.com/LeoLee-likedawn/llm-foundation-lab-project/main/PRJ-T%E1%84%8B%E1%85%AE%E1%84%8C%E1%85%AE%E1%84%89%E1%85%A9%E1%84%80%E1%85%A2-0927.png" alt="rag" align="center" border="0"  width="800" height=auto>
</center>

### 0-2. T우주 서비스 제공 영역별 주요 Q&A
다수의 고객에게 다양한 제휴사의 상품을 제공하는 T우주 서비스의 특성상 다양한 이해관계자를 통해 문의 및 요청이 인입
특히 서비스 출시 준비 단계에서 **1) 채널사/제휴사 기획/개발 담당자**와 **2) QA 담당자**의 문의가 크게 증가하고 서비스 출시가 완료되면 **3) 고객센터**를 통한 실제 고객이나 사용자(대리점/판매점 직원 등)의 문의 및 요청이 증가함
**채널 연동 서비스**의 경우, 주로 이미 **제공된 가이드 문서를 참조**하여 별도 요청 없이 **자체적으로 해결 가능**한 문제가 많으나 정확히 문서를 숙지하지 않은 상태로 문의나 요청이 발생하는 경우가 많음
**제휴사 연동 서비스**의 경우, T우주와 실제 상품을 제공하는 제휴사간 시점마다 발생한 **연동 이력을 확인**(DB 조회)해야 **문의나 요청에 정확하게 댭변이 가능**한 경우가 많음 

<center>
<img src="https://raw.githubusercontent.com/LeoLee-likedawn/llm-foundation-lab-project/main/PRJ-T%E1%84%8B%E1%85%AE%E1%84%8C%E1%85%AEQnA-0927.png" alt="rag" align="center" border="0"  width="800" height=auto>
</center>

### 0-3. 프로젝트 목적 및 수행 범위
**목적** : T우주 외부 제공 서비스 관련 문의 및 요청에 대해 실제 운영자에게 문의나 요청이 전달되기 전에 해결책을 제시하여 **요청자**(고객을 포함한 이해관계자)**의 대기 시간 및 운영 비용 최소화**  
**수행범위**
- 채널 연동 Q&A : 가이드 문서의 주요 문제 발생 원인과 해결책을 기반으로 문제를 해결할 수 있도록 가이드 (전체 채널 연공 관련 문의/요청의 약 90% 해당) 
- 제휴사 연동 Q&A : 특정 고객/계약에 대한 연동 이력을 추출하고 이를 기반으로 문제을 해결할 수 있도록 가이드 (전체 제휴사 연공 관련 문의/요청의 약 90% 해당) 
- 기타 : 정확하게 질문 의도를 파악할 수 없는 경우 사용자가 질문을 스스로 개선할 수 있도록 가이드
![My Logo](PRJ-그래프구조-0928.png)

## 1. 공통/부가 서비스 구현

<center>
<img src="https://raw.githubusercontent.com/LeoLee-likedawn/llm-foundation-lab-project/main/PRJ-%E1%84%80%E1%85%A9%E1%86%BC%E1%84%90%E1%85%A9%E1%86%BC%E1%84%80%E1%85%AE%E1%84%92%E1%85%A7%E1%86%AB-0927.png" alt="rag" align="center" border="0"  width="800" height=auto>
</center>

### 1-1. 공통 환경변수 및 설정 구현

In [93]:
from dotenv import load_dotenv
load_dotenv()

import os
#from glob import glob
from pprint import pprint
#import json
from typing import List, TypedDict

# Langsmith tracing 여부를 확인 
print(os.getenv('LANGSMITH_TRACING'))

DB_TUNIVERSE = 'tuniverse_integration_hist_database.db'
TABLE_TUNIVERSE_AFF_HIST = 'aff_if_hist'
TABLE_TUNIVERSE_MEM_LIST = 'mem_info_list'

class ServiceState(TypedDict):
    service_type: str
    need_to_revise: bool
    user_question: str
    retriever_type: str
    embedding_type: str
    sql_query: str
    search_results: List[str]
    final_answer: str

def create_multiline_text(*args):
    return "\n".join(args)

def merge_state(state: ServiceState) -> str:
    result = create_multiline_text(
        "****************   State Info. ****************",
        f"- service_type : {state["service_type"]}",
        f"- user_question : {state["user_question"]}",
        f"- retriever_type : {state["retriever_type"]}",
        f"- embedding_type : {state["embedding_type"]}",
        f"- sql_query : {state["sql_query"]}",
        f"- search_results : {state["search_results"]}",
        "***************   Final Answer  ***************",
        f"{state["final_answer"]}",
        "***********************************************"
    )
    
    return result

def show_state(state: ServiceState):
    print("-"*33, " State Info.  ", "-"*33)
    print("- state.service_type : ", state["service_type"])
    print("- state.user_question : ", state["user_question"])
    print("- state.retriever_type : ", state["retriever_type"])
    print("- state.embedding_type : ", state["embedding_type"])
    print("- state.sql_query : ", state["sql_query"])
    print("- state.search_results : ", state["search_results"])
    print("-"*33, " Final Answer ", "-"*33)
    print(state["final_answer"])
    print("-"*80)


true


### 프롬프트 등록

In [ ]:
from typing import Literal
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langfuse import get_client
from langfuse.langchain import CallbackHandler

# 콜백 핸들러 생성
langfuse_handler = CallbackHandler()
# Langfuse 클라이언트 초기화
langfuse = get_client()
# 연결 테스트
assert langfuse.auth_check()

#f_use_langfuse = True

def create_prompts_text():
    
    prompt = "사용자가 다음과 같이 질문하였습니다."
    prompt.join()

    langfuse.create_prompt(
        name="service-exam-text-prompt",  # 프롬프트 이름
        type="text",          
        prompt="{{serviceLevel}} 서비스 운영자로서, {{quality}}를 어떻게 생각하시나요?",
        labels=["production"],           # 프로덕션 레이블
        tags=["service", "qa", "text"],    # 태그
        config={
            "model": "gpt-4.1-mini",
            "temperature": 0.7,
            "max_tokens": 500
        }
    )



def create_prompts_chat():
    langfuse.create_prompt(
        name="prompt-generate-response-chat",  # 프롬프트 이름
        type="chat",          
        prompt=[
            {
                "role": "system",
                "content": """
당신은 사용자 입력에 대해 검색 결과를 참고하여 유용한 답변을 제공하는 전문가입니다.
사용자 입력: {user_question}
검색 결과: {search_results}"""
            },
            {
                "role": "system",
                "content": "다음 정보를 참고하여 답변하세요.\n{{search_results}}"
            },
            {
                "role": "user",
                "content": "{{user_question}}"
            }
        ],
        labels=["production"],       # 프로덕션 레이블
        tags=["tuniverse", "project", "chat"],    # 태그
        config={
            "model": "gpt-4.1-mini"
        }
    )

    langfuse.create_prompt(
        name="prompt-check-service-typ-chat",  # 프롬프트 이름
        type="chat",          
        prompt=[
            {
                "role": "system",
                "content": """
당신은 사용자 입력을 통해 질문 영역을 판단하고 정해진 답변을 제공하는 전문가입니다.

질문 영역은 "제휴사" 혹은 "채널" 영역이 존재합니다.
다음 제시된 조건에 하나라도 완전히 부합하는 경우 질문 영역을 판단할 수 있습니다.
제시된 조건은 각각 독립적이며 완전히 부합하는 조건이 하나도 없는 경우 질문 영역을 판단할 수 없고 이 경우 "unknown"으로 답변합니다.
제시된 조건은 우선순위의 내림차순이므로 상위 조건 중 완전히 부합하는 조건이 발견되면 하위 조건은 검사하지 않고 영역을 판단하여 답변합니다.
[조건 1-1] 오류코드(400,401,403,404,500과 같은 399보다 크고 600보다 작은 3자리 숫자)가 존재하는 경우 "채널" 영역으로 판단
[조건 1-2] 오류메시지(Access Restricted, Forbidden, Unauthorized, Bad Request, Not Found, Internal Server Error)가 존재하는 경우 "채널" 영역으로 판단
[조건 2-1] 계약번호(3으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 영역으로 판단 
[조건 2-2] 계약서비스번호(6으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 영역으로 판단 
[조건 2-3] 고객(회원)이름(한글2자이상)과 이동전화번호(010으로 시작하는 13자리 숫자), 상품명(3자리 이상 문자)이 모두 존재하는 경우 "제휴사" 영역으로 판단
[조건 2-4] 고객(회원)번호(1,8,9 로 시작하는 10자리 숫자)와 상품명(3자리 이상 문자)이 모두 존재하는 경우 "제휴사" 영역으로 판단
                
답변:
"제휴사" 영역이라고 판단되면 "affiliate"로 답변 
"채널" 영역이라고 판단되면 "channel"로 답변
아무 조건도 만족하지 않아 판단이 불가하다면 "unknown"으로 답변

사용자 입력: {user_question}
            """
            },            
            {
                "role": "user",
                "content": "{{user_question}}"
            }
        ],
        labels=["production"],       # 프로덕션 레이블
        tags=["tuniverse", "project", "chat"],    # 태그
        config={
            "model": "gpt-4.1-mini"
        }
    )

    langfuse.create_prompt(
        name="prompt-revise-question-chat",  # 프롬프트 이름
        type="chat",          
        prompt=[
            {
                "role": "system",
                "content": """
당신은 사용자가 원하는 결과를 응답받을 수 있도록 사용자 입력을 보완하도록 가이드하는 전문가입니다.

사용자 입력의 다음 키워드 포함 여부로 질문 영역을 추측할 수 있습니다.
- 제휴사 영역 : "고객", ""가입", "해지", "변경", "결제"
- 채널 영역 : "오류", "조회"

질문 영역이 "제휴사" 영역이라고 판단되면 다음 조건에 맞는 정보를 사용자가 질문에 추가할 수 있도록 가이드하세요.
[조건 2-1] 계약번호(3으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 연동 관련 유용한 답변 가능
[조건 2-2] 계약서비스번호(6으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 연동 관련 유용한 답변 가능
[조건 2-3] 고객(회원)이름(한글2자이상)과 이동전화번호(010으로 시작하는 13자리 숫자), 상품명(3자리 이상 문자)이 모두 존재하는 경우 "제휴사" 연동 관련 유용한 답변 가능
[조건 2-4] 고객(회원)번호(1,8,9 로 시작하는 10자리 숫자)와 상품명(3자리 이상 문자)이 모두 존재하는 경우 "제휴사" 연동 관련 유용한 답변 가능

질문 영역이 "채널" 영역이라고 판단되면 다음 조건에 맞는 정보를 사용자가 질문에 추가할 수 있도록 가이드하세요.
[조건 1-1] 오류코드(400,401,403,404,500과 같은 399보다 크고 600보다 작은 3자리 숫자)가 존재하는 경우 "채널" 영역으로 판단
[조건 1-2] 오류메시지(Access Restricted, Forbidden, Unauthorized, Bad Request, Not Found, Internal Server Error)가 존재하는 경우 "채널" 영역으로 판단

질문 영역을 판단할 수 없다면 질문 영역별로 위 제시된 조건에 맞는 정보를 질문에 추가할 수 있도록 가이드하세요. (제휴사, 채널 영역 분리하여 모두 가이드)

사용자 입력: {user_question}
                """
            },            
            {
                "role": "user",
                "content": "{{user_question}}"
            }
        ],
        labels=["production"],       # 프로덕션 레이블
        tags=["tuniverse", "project", "chat"],    # 태그
        config={
            "model": "gpt-4.1-mini"
        }
    )

create_prompts_chat()
#create_prompts_text()



# 프로덕션 버전 가져오기
prompt = langfuse.get_prompt("prompt-generate-response-chat")

# 프롬프트 정보 출력
print(f"모델: {prompt.config['model']}")
print(f"라벨: {prompt.labels}")
print(f"태그: {prompt.tags}")
print(f"프롬프트: {prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(prompt.get_langchain_prompt())

### 1-2. Q&A 영역 분류 서비스 구현
### 1-3. 질의 보완 서비스 구현
### 1-4. 최종 답변 서비스 구현

In [ ]:
from typing import Literal
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_core.prompts import PromptTemplate

# 콜백 핸들러 생성
langfuse_handler = CallbackHandler()
# Langfuse 클라이언트 초기화
langfuse = get_client()
# 연결 테스트
assert langfuse.auth_check()

#로그 출력 여부
f_print_log = True
#Lanfuse 사용 여부
f_use_langfuse = True

# LLM 인스턴스 생성
llm = ChatOpenAI(model="gpt-4.1-mini")

### 1-2. Q&A 영역 분류 서비스 구현
def chk_svc_typ(state: ServiceState) -> ServiceState:
    print("=============== chk_svc_typ ===============")
    """ 사용자 입력이 질문 영역(제휴사/채널)에 대해 판단하는 함수 """    

    # 사용자의 문의 영역을 분석하기 위한 템플릿
    analyze_template = """
    사용자 입력: {user_question}
    
    당신은 사용자 입력을 통해 질문 영역을 판단하고 정해진 답변을 제공하는 전문가입니다.
    질문 영역은 "제휴사" 혹은 "채널" 영역이 존재합니다.

    다음 제시된 조건에 하나라도 완전히 부합하는 경우 질문 영역을 판단할 수 있습니다. 반드시 조건에 부합하는지 확인해야 합니다.
    제시된 조건은 각각 독립적이며 완전히 부합하는 조건이 하나도 없는 경우 질문 영역을 판단할 수 없고 이 경우 "unknown"으로 답변합니다.
    제시된 조건은 우선순위의 내림차순이므로 상위 조건 중 완전히 부합하는 조건이 발견되면 하위 조건은 검사하지 않고 영역을 판단하여 답변합니다.
    
    [조건 1-1] 오류코드(400,401,403,404,500과 같은 399보다 크고 600보다 작은 3자리 숫자)가 존재하는 경우 "채널" 영역으로 판단
    [조건 1-2] 오류메시지(Access Restricted, Forbidden, Unauthorized, Bad Request, Not Found, Internal Server Error)가 존재하는 경우 "채널" 영역으로 판단
    [조건 2-1] 계약번호(3으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 영역으로 판단 
    [조건 2-2] 계약서비스번호(6으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 영역으로 판단 
    [조건 2-3] 고객(회원)이름(한글2자이상)과 이동전화번호(010으로 시작하는 "-"제외한 11자리 숫자), 상품명(3자리 이상 문자)이 모두 존재하는 경우 "제휴사" 영역으로 판단
    [조건 2-4] 고객(회원)번호(1,8,9 로 시작하는 10자리 숫자)와 상품명(3자리 이상 문자)이 모두 존재하는 경우 "제휴사" 영역으로 판단

    [답변형식]
    "제휴사" 영역이라고 판단되면 "affiliate"로 답변 
    "채널" 영역이라고 판단되면 "channel"로 답변
    아무 조건도 만족하지 않아 판단이 불가하다면 "unknown"으로 답변
    """

    if f_use_langfuse:
        print("Use lanfuse!!! chk_svc_type")
        
        chat_prompt = langfuse.get_prompt("prompt-check-service-typ-chat", type="chat")

        print("Use lanfuse!!! #0 : ")  
        langchain_prompt = ChatPromptTemplate.from_messages(
            chat_prompt.get_langchain_prompt()
        )
        print("Use lanfuse!!! #1 : ", langchain_prompt)
        langchain_prompt.metadata = {"langfuse_prompt": chat_prompt}   # Langfuse 자동 링크를 위한 메타데이터

        # 체인 생성 및 실행
        print("Use lanfuse!!! #2 : ")
        chain = langchain_prompt | llm
        result = chain.invoke(
            input={"user_question": state['user_question']},
            config={"callbacks": [langfuse_handler]}  # Langfuse 트레이싱을 위한 콜백
        )
        print("Use lanfuse!!! #3 : ", result)
        service_type = result
        
    else:
        print("Not use lanfuse!!!")
        analyze_prompt = ChatPromptTemplate.from_template(analyze_template)

        analyze_chain = analyze_prompt | llm | StrOutputParser()
        result = analyze_chain.invoke({"user_question": state['user_question']})
        service_type = result.strip().lower()

    # 사용자 입력을 분석하여 한국어인지 판단
        
    state['service_type'] = service_type
    if f_print_log:
        print ("SVC TYPE : ", state['service_type'])

    # 결과를 상태에 업데이트 
    return state


def decide_next_step(
        state: ServiceState
    ) -> Literal["aff_svc_qna", "chn_svc_qna", "revise_qna"]:#, "generate_response"]:
    """ 다음 실행 단계를 결정하는 함수 """

    # 한국어인 경우 한국어 문서 검색 함수 실행
    if state['service_type'] == "affiliate":
        return "aff_svc_qna"
    elif state['service_type'] == "channel":
        return "chn_svc_qna"
    else:
        return "revise_qna"  

### 1-3. 질의 보완 서비스 구현
def revise_qna(state: ServiceState) -> ServiceState:
    """ 사용자 입력이 질문 영역(제휴사/채널)을 판단하기 부족한 경우 입력 보완 방안을 제공하는 함수 """

    # 사용자의 문의 영역을 분석하기 위한 템플릿
    revise_templet = """
    사용자 입력: {user_question}

    당신은 사용자가 원하는 결과를 응답받을 수 있도록 사용자 입력을 보완하도록 가이드하는 전문가입니다.

    사용자 입력의 다음 키워드 포함 여부로 질문 영역을 추측할 수 있습니다.
    - 제휴사 영역 : "고객", ""가입", "해지", "변경", "결제"
    - 채널 영역 : "오류", "조회"

    질문 영역이 "제휴사" 영역이라고 판단되면 다음 조건에 맞는 정보를 사용자가 질문에 추가할 수 있도록 구체적으로 가이드하세요.
    [조건 2-1] 계약번호(3으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 연동 관련 유용한 답변 가능
    [조건 2-2] 계약서비스번호(6으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 연동 관련 유용한 답변 가능
    [조건 2-3] 고객(회원)이름(한글2자이상)과 이동전화번호(010으로 시작하는 "-"제외한 11자리 숫자), 상품명(3자리 이상 문자)이 모두 존재하는 경우 "제휴사" 연동 관련 유용한 답변 가능
    [조건 2-4] 고객(회원)번호(1,8,9 로 시작하는 10자리 숫자)와 상품명(3자리 이상 문자)이 모두 존재하는 경우 "제휴사" 연동 관련 유용한 답변 가능

    질문 영역이 "채널" 영역이라고 판단되면 다음 조건에 맞는 정보를 사용자가 질문에 추가할 수 있도록 구체적으로 가이드하세요.
    [조건 1-1] 오류코드(400,401,403,404,500과 같은 399보다 크고 600보다 작은 3자리 숫자)가 존재하는 경우 "채널" 영역으로 판단
    [조건 1-2] 오류메시지(Access Restricted, Forbidden, Unauthorized, Bad Request, Not Found, Internal Server Error)가 존재하는 경우 "채널" 영역으로 판단

    질문 영역을 판단할 수 없다면 질문 영역별로 위 제시된 조건에 맞는 정보를 질문에 추가할 수 있도록 구체적으로 가이드하세요. (제휴사, 채널 영역 분리하여 모두 가이드)
    """

    if f_use_langfuse:
        print("Use lanfuse!!!")
        chat_prompt = langfuse.get_prompt("prompt-revise-question-chat", type="chat")
        
        langchain_prompt = ChatPromptTemplate.from_messages(
            chat_prompt.get_langchain_prompt()
        )
        langchain_prompt.metadata = {"langfuse_prompt": chat_prompt}   # Langfuse 자동 링크를 위한 메타데이터

        # 체인 생성 및 실행
        chain = langchain_prompt | llm
        result = chain.invoke(
            input={"user_question": state['user_question']},
            config={"callbacks": [langfuse_handler]}  # Langfuse 트레이싱을 위한 콜백
        )
        final_answer = result.strip().lower()

    else:
        print("Not use lanfuse!!!")
        revise_prompt = ChatPromptTemplate.from_template(revise_templet)
        analyze_chain = revise_prompt | llm | StrOutputParser()
        result = analyze_chain.invoke({"user_question": state['user_question']})
        final_answer = result.strip().lower()    
    
    state['final_answer'] = final_answer
    if f_print_log:
        print ("final_answer : ", state['final_answer'])
   
    # 결과를 상태에 업데이트 
    return state

### 1-4. 최종 답변 서비스 구현
def gen_res(state: ServiceState) -> ServiceState:
    """ 답변 생성 함수 """

    # 답변 템플릿
    response_template = """
    당신은 사용자 입력에 대해 검색 결과를 참고하여 유용한 답변을 제공하는 전문가입니다.
    사용자 입력: {user_question}
    검색 결과: {search_results} 
    """

    if f_use_langfuse:
        print("Use lanfuse!!!")
        chat_prompt = langfuse.get_prompt("prompt-generate-response-chat", type="chat")
        
        langchain_prompt = ChatPromptTemplate.from_messages(
            chat_prompt.get_langchain_prompt()
        )
        langchain_prompt.metadata = {"langfuse_prompt": chat_prompt}   # Langfuse 자동 링크를 위한 메타데이터

        # 체인 생성 및 실행
        chain = langchain_prompt | llm
        final_answer = chain.invoke(
            input={"user_question": state['user_question'], "search_results": state['search_results']},
            config={"callbacks": [langfuse_handler]}  # Langfuse 트레이싱을 위한 콜백
        )        

    else:
        print("Not use lanfuse!!!")
        response_prompt = ChatPromptTemplate.from_template(response_template)
        response_chain = response_prompt | llm | StrOutputParser()
        
        final_answer = response_chain.invoke(
            {
                "user_question": state['user_question'],   # 사용자 입력 (상태에서 가져옴)
                "search_results": state['search_results'],  # 검색 결과 (상태에서 가져옴)                
            }
        )
    
    state['final_answer'] = final_answer
    if f_print_log:
        print ("final_answer : ", state['final_answer'])

    # 결과를 상태에 저장
    return state

## 2. 제휴사 연동 서비스 Q&A 구현

### 2-1. 제휴사 Q&A 데이터 구성

In [ ]:
import pandas as pd
import sqlite3

#로그 출력 여부
f_print_log = True

aff_hist_file = "data/PRJ_AFFC_HIST_EN_CSV.csv"

#df_qa_test = pd.read_excel("data/PRJ_AFFC_HIST_XLS.xls")
aff_hist_data = pd.read_csv(aff_hist_file)

print(f"load completed...FILE[{aff_hist_file}] CNT[{aff_hist_data.shape[0]}]")
aff_hist_data.head(2)
print("-"*80)


aff_hist_data.columns = ['AUDIT_DTM', 'AUDIT_ID', 'MBR_NUM', 'MBR_NM', 'CTR_NUM', 'CTR_SVC_NUM', 'AFFC_BZR_NM', 'AFFC_LNKG_TSK_CD', 'AFFC_LNKG_TRMS_CD', 'TRMS_RSLT_CD', 'TRMS_ERR_CD', 'TRMS_ERR_MSG_CTT', 'TRMS_REQ_CNTT', 'TRMS_RES_CNTT', 'PRD_ID', 'PRD_NM', 'SKU_ID', 'SKU_NM']
# 데이터 타입 변환
def convert_to_numeric_safely(value):
    try:
        return pd.to_numeric(value)
    except:
        return None

def convert_to_datetime_safely(value):
    try:
        return pd.to_datetime(value)
    except:
        return None

# 각 컬럼의 데이터 타입 변환
aff_hist_data['AUDIT_DTM'] = aff_hist_data['AUDIT_DTM'].apply(convert_to_datetime_safely)
aff_hist_data['AUDIT_ID'] = aff_hist_data['AUDIT_ID'].apply(lambda x: str(x).strip())
aff_hist_data['MBR_NUM'] = aff_hist_data['MBR_NUM'].apply(lambda x: str(x).strip())
aff_hist_data['MBR_NM'] = aff_hist_data['MBR_NM'].apply(lambda x: str(x).strip())
aff_hist_data['CTR_NUM'] = aff_hist_data['CTR_NUM'].apply(lambda x: str(x).strip())
aff_hist_data['CTR_SVC_NUM'] = aff_hist_data['CTR_SVC_NUM'].apply(lambda x: str(x).strip())
aff_hist_data['AFFC_BZR_NM'] = aff_hist_data['AFFC_BZR_NM'].apply(lambda x: str(x).strip())
aff_hist_data['AFFC_LNKG_TSK_CD'] = aff_hist_data['AFFC_LNKG_TSK_CD'].apply(lambda x: str(x).strip())
aff_hist_data['AFFC_LNKG_TRMS_CD'] = aff_hist_data['AFFC_LNKG_TRMS_CD'].apply(lambda x: str(x).strip())
aff_hist_data['TRMS_RSLT_CD'] = aff_hist_data['TRMS_RSLT_CD'].apply(lambda x: str(x).strip())
aff_hist_data['TRMS_ERR_CD'] = aff_hist_data['TRMS_ERR_CD'].apply(lambda x: str(x).strip())
aff_hist_data['TRMS_ERR_MSG_CTT'] = aff_hist_data['TRMS_ERR_MSG_CTT'].apply(lambda x: str(x).strip())
aff_hist_data['TRMS_REQ_CNTT'] = aff_hist_data['TRMS_REQ_CNTT'].apply(lambda x: str(x).strip())
aff_hist_data['TRMS_RES_CNTT'] = aff_hist_data['TRMS_RES_CNTT'].apply(lambda x: str(x).strip())
aff_hist_data['PRD_ID'] = aff_hist_data['PRD_ID'].apply(lambda x: str(x).strip())
aff_hist_data['PRD_NM'] = aff_hist_data['PRD_NM'].apply(lambda x: str(x).strip())
aff_hist_data['SKU_ID'] = aff_hist_data['SKU_ID'].apply(lambda x: str(x).strip())
aff_hist_data['SKU_NM'] = aff_hist_data['SKU_NM'].apply(lambda x: str(x).strip())

print("convert type of column completed...")
aff_hist_data.info()
print("-"*80)



# SQLite 데이터베이스 생성
conn = sqlite3.connect(DB_TUNIVERSE)
cursor = conn.cursor()

if f_print_log:
    print(f"connect database completed...DB[{DB_TUNIVERSE}]")
    print("-"*80)

# 테이블 삭제 (if exists)
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_TUNIVERSE_AFF_HIST}")

# 테이블 생성
cursor.execute("""
CREATE TABLE aff_if_hist (
    AUDIT_DTM DATETIME,
    AUDIT_ID TEXT,
    MBR_NUM TEXT,
    MBR_NM TEXT,
    CTR_NUM TEXT,
    CTR_SVC_NUM TEXT,
    AFFC_BZR_NM TEXT,
    AFFC_LNKG_TSK_CD TEXT,
    AFFC_LNKG_TRMS_CD TEXT,
    TRMS_RSLT_CD TEXT,
    TRMS_ERR_CD TEXT,
    TRMS_ERR_MSG_CTT TEXT,
    TRMS_REQ_CNTT TEXT,
    TRMS_RES_CNTT TEXT,
    PRD_ID TEXT,
    PRD_NM TEXT,
    SKU_ID TEXT,
    SKU_NM TEXT
)
""")

if f_print_log:
    print(f"create table completed...TBL[{TABLE_TUNIVERSE_AFF_HIST}]")
    print("--- table info. ---")
    aff_hist_data.info()
    print("-"*80)

# 데이터 삽입
for _, row in aff_hist_data.iterrows():
    try:
        cursor.execute(f"""
        INSERT INTO {TABLE_TUNIVERSE_AFF_HIST} VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            str(row['AUDIT_DTM']),
            str(row['AUDIT_ID']),
            str(row['MBR_NUM']),
            str(row['MBR_NM']),
            str(row['CTR_NUM']),
            str(row['CTR_SVC_NUM']),
            str(row['AFFC_BZR_NM']),
            str(row['AFFC_LNKG_TSK_CD']),
            str(row['AFFC_LNKG_TRMS_CD']),
            str(row['TRMS_RSLT_CD']),
            str(row['TRMS_ERR_CD']),
            str(row['TRMS_ERR_MSG_CTT']),
            str(row['TRMS_REQ_CNTT']),
            str(row['TRMS_RES_CNTT']),
            str(row['PRD_ID']),
            str(row['PRD_NM']),
            str(row['SKU_ID']),
            str(row['SKU_NM'])
        ))
    except Exception as e:
        print(f"Error inserting row: {row}")
        print(f"Error message: {str(e)}")
        continue

# 변경사항 저장
conn.commit()

# 데이터베이스 상태 확인
cursor.execute("SELECT COUNT(*) FROM aff_if_hist")
aff_if_hist_count = cursor.fetchone()[0]

if f_print_log:
    print(f"insert table completed...TOT_CNT[{aff_if_hist_count}]")
    cursor.execute(f"SELECT * FROM {TABLE_TUNIVERSE_AFF_HIST} LIMIT 3")
    print("--- sample data ---")
    for fetch in cursor.fetchall():
        print("- ", fetch)
    print("-"*80)

load completed...FILE[data/PRJ_AFFC_HIST_EN_CSV.csv] CNT[148]
--------------------------------------------------------------------------------
convert type of column completed...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   AUDIT_DTM          148 non-null    datetime64[ns]
 1   AUDIT_ID           148 non-null    object        
 2   MBR_NUM            148 non-null    object        
 3   MBR_NM             148 non-null    object        
 4   CTR_NUM            148 non-null    object        
 5   CTR_SVC_NUM        148 non-null    object        
 6   AFFC_BZR_NM        148 non-null    object        
 7   AFFC_LNKG_TSK_CD   148 non-null    object        
 8   AFFC_LNKG_TRMS_CD  148 non-null    object        
 9   TRMS_RSLT_CD       148 non-null    object        
 10  TRMS_ERR_CD        148 non-null    object        
 

In [64]:
import pandas as pd
import sqlite3

#로그 출력 여부
f_print_log = True

aff_hist_file = "data/PRJ_MEM_LISTS_CSV.csv"
aff_hist_data = pd.read_csv(aff_hist_file)

print(f"load completed...FILE[{aff_hist_file}] CNT[{aff_hist_data.shape[0]}]")
aff_hist_data.head(2)
print("-"*80)

#MBR_NUM	
#MBR_NM	
#MPHON_NUM	
#EMIL_ADDR

aff_hist_data.columns = ['MBR_NUM', 'MBR_NM', 'MPHON_NUM', 'EMIL_ADDR']
# 데이터 타입 변환
def convert_to_numeric_safely(value):
    try:
        return pd.to_numeric(value)
    except:
        return None

def convert_to_datetime_safely(value):
    try:
        return pd.to_datetime(value)
    except:
        return None

# 각 컬럼의 데이터 타입 변환
aff_hist_data['MBR_NUM'] = aff_hist_data['MBR_NUM'].apply(lambda x: str(x).strip())
aff_hist_data['MBR_NM'] = aff_hist_data['MBR_NM'].apply(lambda x: str(x).strip())
aff_hist_data['MPHON_NUM'] = aff_hist_data['MPHON_NUM'].apply(lambda x: str(x).strip())
aff_hist_data['EMIL_ADDR'] = aff_hist_data['EMIL_ADDR'].apply(lambda x: str(x).strip())

print("convert type of column completed...")
aff_hist_data.info()
print("-"*80)

# SQLite 데이터베이스 생성
conn = sqlite3.connect(DB_TUNIVERSE)
cursor = conn.cursor()

if f_print_log:
    print(f"connect database completed...DB[{DB_TUNIVERSE}]")
    print("-"*80)

# 테이블 삭제 (if exists)
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_TUNIVERSE_MEM_LIST}")

# 테이블 생성
cursor.execute("""
CREATE TABLE mem_info_list (
    MBR_NUM TEXT,
    MBR_NM TEXT,
    MPHON_NUM TEXT,
    EMIL_ADDR TEXT
)
""")

if f_print_log:
    print(f"create table completed...TBL[{TABLE_TUNIVERSE_MEM_LIST}]")
    print("--- table info. ---")
    aff_hist_data.info()
    print("-"*80)

# 데이터 삽입
for _, row in aff_hist_data.iterrows():
    try:
        cursor.execute(f"""
        INSERT INTO {TABLE_TUNIVERSE_MEM_LIST} VALUES (?, ?, ?, ?)
        """, (
            str(row['MBR_NUM']),
            str(row['MBR_NM']),
            str(row['MPHON_NUM']),
            str(row['EMIL_ADDR'])
        ))
    except Exception as e:
        print(f"Error inserting row: {row}")
        print(f"Error message: {str(e)}")
        continue

# 변경사항 저장
conn.commit()

# 데이터베이스 상태 확인
cursor.execute("SELECT COUNT(*) FROM mem_info_list")
aff_if_hist_count = cursor.fetchone()[0]

if f_print_log:
    print(f"insert table completed...TOT_CNT[{aff_if_hist_count}]")
    cursor.execute(f"SELECT * FROM {TABLE_TUNIVERSE_MEM_LIST} LIMIT 3")
    print("--- sample data ---")
    for fetch in cursor.fetchall():
        print("- ", fetch)
    print("-"*80)

load completed...FILE[data/PRJ_MEM_LISTS_CSV.csv] CNT[6]
--------------------------------------------------------------------------------
convert type of column completed...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   MBR_NUM    6 non-null      object
 1   MBR_NM     6 non-null      object
 2   MPHON_NUM  6 non-null      object
 3   EMIL_ADDR  6 non-null      object
dtypes: object(4)
memory usage: 324.0+ bytes
--------------------------------------------------------------------------------
connect database completed...DB[tuniverse_integration_hist_database.db]
--------------------------------------------------------------------------------
create table completed...TBL[mem_info_list]
--- table info. ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  -----

### 2-2. 제휴사 연동 서비스 구현

In [116]:
from typing import List, TypedDict
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain.chains import create_sql_query_chain
from langchain_core.prompts import ChatPromptTemplate
from typing import Annotated, TypedDict
from langchain_core.prompts import ChatPromptTemplate
from typing_extensions import TypedDict, Annotated
from langchain_community.tools import QuerySQLDatabaseTool
from langchain.agents import tool
from langchain.agents import create_react_agent

# 로그 출력 여부
f_print_log = True

# 에이전트 사용 여부
f_use_agent = True

# LLM 인스턴스 생성
llm = ChatOpenAI(model="gpt-4.1-mini")

class QueryOutput(TypedDict):
    """Generated SQL query."""
    query: Annotated[str, ..., "Syntactically valid SQL query."]
    result: Annotated[str, ..., "Result of the query."]

@tool
def user_search(user_question: str) -> str:
    """ 고객(회원)의 정확한 고객(회원) 번호를 추출"""   

    query_prompt_template = ChatPromptTemplate.from_messages([
        ("system", 
        """
        사용자 입력: {user_question}
        대상 테이블 : "mem_info_list"
        
        당신은 사용자 입력을 통해 기본적인 정보를 확인하고 이를 이용해 DB의 대상 테이블에서 고객(회원) 번호를 조회하는 쿼리를 작성하는 도구입니다.
        올바른 {dialect} 쿼리를 작성하고 반환하세요.        

        사용자 입력에는 다음 정보가 있을 수 있습니다.
        - 고객번호(회원번호) : 형식 - 1,8,9 로 시작하는 10자리 숫자
        - 고객명(회원명) : 형식 - 한글2자이상
        - 전화번호 : 형식 - 010으로 시작하는 13자리 숫자 ("-"은 무시)

        "mem_info_list" 테이블에는 다음 컬럼과 정보들이 있습니다. 
        - MBR_NUM : 고객번호(회원번호) - 1,8,9 로 시작하는 10자리 숫자
        - MBR_NM : 고객명(회원명) - 한글2자이상
        - MPHON_NUM : 전화번호 - 010으로 시작하는 13자리 숫자 ("-"은 무시)
        - EMIL_ADDR : 이메일주소 - 이메일 형식

        사용자 입력에 있는 정보를 기반으로 "mem_info_list" 테이블에서 고객번호(회원번호)를 조회하는 쿼리를 반환하세요.        
        """),
        ("user", 
        """
        Question:
        {user_question}
        """)
    ])

    query_prompt_template.input_schema.model_json_schema()

    # SQLite 데이터베이스 연결
    db = SQLDatabase.from_uri(f"sqlite:///{DB_TUNIVERSE}")

    if f_print_log:
        print(f"conect database...DB[{DB_TUNIVERSE}]")
        # 사용 가능한 테이블 목록 출력
        print("--- table list ---")
        tables = db.get_usable_table_names()
        for table in tables:
            print("- ",table)
        print("-"*80)    

    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "user_question": user_question,
            "dialect": db.dialect,
            "table_info": db.get_table_info(),
        }
    )    
    structured_llm = llm.with_structured_output(QueryOutput)
    sql_query = structured_llm.invoke(prompt)
    
    print(f"RES_QRY[{sql_query["query"]}]")    

    execute_query_tool = QuerySQLDatabaseTool(db=db)
    search_results = execute_query_tool.invoke(sql_query)

    search_results = "고객번호(mbr_num) : " + search_results
    if f_print_log:
        print ("search_results : ", search_results)

    return search_results

@tool
def hist_search(user_question: str, mbr_num: str) -> List:
    """ 고객(회원)의 상품 사용 이력을 추출 """   

    search_results = []

    query_prompt_template = ChatPromptTemplate.from_messages([
        ("system", 
        """
        사용자 입력: {user_question}
        고객(회원) 번호 : {mbr_num}
        대상 테이블 : "aff_if_hist"
        
        당신은 사용자 입력을 통해 기본적인 정보를 확인하고 이를 이용해 DB의 대상 테이블에서 고객(회원)의 상품 사용 이력을 조회하는 쿼리를 작성하는 도구입니다.
        올바른 {dialect} 쿼리를 작성하고 반환하세요.        

        "aff_if_hist" 테이블에는 다음 컬럼과 정보들이 있습니다.
        - AUDIT_DTM : 연동이력 발생일시
        - AUDIT_ID : 처리자ID
        - MBR_NUM : 고객번호
        - MBR_NM : 고객명
        - CTR_NUM : 계약번호
        - CTR_SVC_NUM : 계약서비스번호
        - AFFC_BZR_NM : 제휴사명 - 상품을 제공하는 회사명으로 고객이 인지하는 정보와 정확하게 일치하지 않을 수 있으니 유사한 값으로 조회 필요
        - AFFC_LNKG_TSK_CD : 업무유형
        --- Q2 : 자동인증가능여부조회
        --- A1 : 가입준비
        --- A2 : 가입
        --- Q3 : 해지가능여부조회
        --- Z1 : 해지
        --- Z3 : 해지취소
        --- CO : 계약연장
        --- CN : 연장취소
        - AFFC_LNKG_TRMS_CD : 연동유형
        --- SB_RQ : T우주 처리 요청
        --- SB_RS : T우주 처리 결과 응답
        --- BS_RQ : 제휴사 처리 요청
        --- BS_RS : 제휴사 처리 결과 응답
        --- BS_AF : 제휴사 처리 결과 T우주 내부 전달
        - TRMS_RSLT_CD : 결과코드
        --- S : 성공
        --- F : 실패
        - TRMS_ERR_CD : 에러코드
        - TRMS_ERR_MSG_CTT : 에러메시지
        - TRMS_REQ_CNTT : 요청전문 - 요청에 사용된 JSON 형태의 전문
        - TRMS_RES_CNTT : 응답전문 - 요청에 대한 응답으로 사용된 JSON 형태의 전문
        - PRD_ID : 패키지ID - 실제 고객이 사용하는 단위 상품들의 묶음 상품의 ID로 10자리 문자열(PR로 시작)로 조회시 정확하게 일치해야 함
        - PRD_NM : 패키지명 - 계약번호와 연결되는 실제 고객이 사용하는 단위 상품들의 묶음 상품명(패키지 상품명)으로 고객이 인지하는 정보와 정확하게 일치하지 않을 수 있으니 유사한 값으로 조회 필요
        - SKU_ID : 상품ID - 실제 고객이 사용하는 단위 상품의 ID로 10자리 문자열(SU로 시작)로 조회시 정확하게 일치해야 함
        - SKU_NM : 상품명 - 계약서비스번호와 매핑되는 실제 고객이 사용하는 단위 상품명으로 고객이 인지하는 정보와 정확하게 일치하지 않을 수 있으니 유사한 값으로 조회 필요

        사용자 입력에 있는 정보를 기반으로 "aff_if_hist" 테이블에서 고객번호(회원번호)를 조회하는 쿼리를 반환하세요.        
        """),
        ("user", 
        """
        Question:
        {user_question}
        """)
    ])

    query_prompt_template.input_schema.model_json_schema()

    # SQLite 데이터베이스 연결
    db = SQLDatabase.from_uri(f"sqlite:///{DB_TUNIVERSE}")

    if f_print_log:
        print(f"conect database...DB[{DB_TUNIVERSE}]")
        # 사용 가능한 테이블 목록 출력
        print("--- table list ---")
        tables = db.get_usable_table_names()
        for table in tables:
            print("- ",table)
        print("-"*80)    

    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "user_question": user_question,
            "mbr_num": mbr_num,
            "dialect": db.dialect,
            "table_info": db.get_table_info(),
        }
    )    
    structured_llm = llm.with_structured_output(QueryOutput)
    sql_query = structured_llm.invoke(prompt)
    
    print(f"RES_QRY[{sql_query["query"]}]")    

    execute_query_tool = QuerySQLDatabaseTool(db=db)
    search_results = execute_query_tool.invoke(sql_query)

    if f_print_log:
        print ("search_results : ", search_results)

    return search_results

def aff_svc_qna_using_agent(state: ServiceState) -> ServiceState:
    """ 채널 연동 관련 오류 원인을 조사하는 함수 """
    
    # 도구 목록
    tools = [user_search, hist_search]  

    # OpenAI GPT-4.1-mini 모델 사용
    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

    # 도구 실행 에이전트 생성
    search_agent = create_react_agent(llm, tools=tools)

    # 도구 실행 에이전트 사용
    search_results = search_agent.invoke(
        {"messages": [
            ("system", 
            """
            당신은 사용자 입력을 기반으로 사용자의 상품 사용 이력을 조사하고 제공하는 에이전트입니다.
            주어진 도구를 활용하여 사용자가 필요한 정보를 찾아서 제공해야 합니다.
            
            [도구(tool) 사용 전략]
            - user_search : 고객(회원)번호(mbr_num)이 분명하지 않은 경우 사용 (고객명(mbr_nm)와 전화번호(mphon_num) 정보가 있는 경우 사용
            - hist_search : 고객(회원)번호(mbr_num)와 상품정보가 함께 존재하거나 계약번호 혹은 계약서비스번호가 존재하는 경우 사용
            - "user_search"는 필요시 선택적으로 사용하고 "hist_search"는 마지막에 필수적으로 사용해야 합니다.
            """),
            ("human", "{user_question}에 대한 이력 정보를 제공해 주세요.")
            ]
        }
    )
    state['search_results'] = search_results
    if f_print_log:
        print ("search_results : ", state['search_results'])

    # 결과를 상태에 업데이트 
    return state


def aff_svc_qna_simple(state: ServiceState) -> ServiceState:

    query_prompt_template = ChatPromptTemplate.from_messages([
        ("system", 
        """
        당신은 SQL 데이터베이스와 상호작용하도록 설계된 에이전트입니다.
        입력된 질문을 기반으로 구문적으로 올바른 {dialect} 쿼리를 작성하고 실행한 뒤, 실행 결과를 답변으로 반환해야 합니다.
        
        입력된 질문으로 쿼리를 작성할 때에는 반드시 아래에서 제공하는 컬럼명과 컬럼값에 의미와 참고사항을 고려하여 작성해야 합니다.
        "-" 기호 뒤에 나오는 값은 테이블의 컬럼명과 그 의미와 참고사항입니다. (예시 : - 컬럼명 : 의미 - 참고사항)
        "---" 기호는 바로 위 컬럼의 값으로 등장할 수 있는 값과 그 의미입니다. (예시 : --- 컬럼값 : 의미 - 참고사항)
        - AUDIT_DTM : 연동이력 발생일시
        - AUDIT_ID : 처리자ID
        - MBR_NUM : 고객번호
        - MBR_NM : 고객명
        - CTR_NUM : 계약번호
        - CTR_SVC_NUM : 계약서비스번호
        - AFFC_BZR_NM : 제휴사명 - 상품을 제공하는 회사명으로 고객이 인지하는 정보와 정확하게 일치하지 않을 수 있으니 유사한 값으로 조회 필요
        - AFFC_LNKG_TSK_CD : 업무유형
        --- Q2 : 자동인증가능여부조회
        --- A1 : 가입준비
        --- A2 : 가입
        --- Q3 : 해지가능여부조회
        --- Z1 : 해지
        --- Z3 : 해지취소
        --- CO : 계약연장
        --- CN : 연장취소
        - AFFC_LNKG_TRMS_CD : 연동유형
        --- SB_RQ : T우주 처리 요청
        --- SB_RS : T우주 처리 결과 응답
        --- BS_RQ : 제휴사 처리 요청
        --- BS_RS : 제휴사 처리 결과 응답
        --- BS_AF : 제휴사 처리 결과 T우주 내부 전달
        - TRMS_RSLT_CD : 결과코드
        --- S : 성공
        --- F : 실패
        - TRMS_ERR_CD : 에러코드
        - TRMS_ERR_MSG_CTT : 에러메시지
        - TRMS_REQ_CNTT : 요청전문 - 요청에 사용된 JSON 형태의 전문
        - TRMS_RES_CNTT : 응답전문 - 요청에 대한 응답으로 사용된 JSON 형태의 전문
        - PRD_ID : 패키지ID - 실제 고객이 사용하는 단위 상품들의 묶음 상품의 ID로 10자리 문자열(PR로 시작)로 조회시 정확하게 일치해야 함
        - PRD_NM : 패키지명 - 계약번호와 연결되는 실제 고객이 사용하는 단위 상품들의 묶음 상품명(패키지 상품명)으로 고객이 인지하는 정보와 정확하게 일치하지 않을 수 있으니 유사한 값으로 조회 필요
        - SKU_ID : 상품ID - 실제 고객이 사용하는 단위 상품의 ID로 10자리 문자열(SU로 시작)로 조회시 정확하게 일치해야 함
        - SKU_NM : 상품명 - 계약서비스번호와 매핑되는 실제 고객이 사용하는 단위 상품명으로 고객이 인지하는 정보와 정확하게 일치하지 않을 수 있으니 유사한 값으로 조회 필요

        반환되는 결과의 컬럼명이나 컬럼의 값이 제공된 실제 의미로 대체 가능한 경우 해당 값을 대체하여 결과를 반환해야 합니다.
        
        작업을 시작할 때는 항상 데이터베이스의 테이블 목록을 확인해야 합니다.
        이 단계를 건너뛰지 마십시오.
        {table_info}
        """),
        ("user", 
        """
        Question:
        {query}
        """)
    ])

    query_prompt_template.input_schema.model_json_schema()

    # SQLite 데이터베이스 연결
    db = SQLDatabase.from_uri(f"sqlite:///{DB_TUNIVERSE}")

    if f_print_log:
        print(f"conect database...DB[{DB_TUNIVERSE}]")
        # 사용 가능한 테이블 목록 출력
        print("--- table list ---")
        tables = db.get_usable_table_names()
        for table in tables:
            print("- ",table)
        print("-"*80)    

    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "query": state["user_question"],
            "dialect": db.dialect,
            "table_info": db.get_table_info(),
        }
    )
    #print("!!! prompt : ", prompt)
    structured_llm = llm.with_structured_output(QueryOutput)

    sql_query = structured_llm.invoke(prompt)
    print(f"RES_QRY[{sql_query["query"]}]")

    state['sql_query'] = sql_query

    execute_query_tool = QuerySQLDatabaseTool(db=db)

    search_results = execute_query_tool.invoke(state)

    state['search_results'] = search_results
    if f_print_log:
        print ("search_results : ", state['search_results'])

    return state

def aff_svc_qna(state: ServiceState) -> ServiceState:
    """ 제휴사 연동 관련 오류 원인을 조사하는 함수 """
    
    if f_use_agent:
        if f_print_log:
            print("--- aff_svc_qna_using_agent ---")
        state =aff_svc_qna_using_agent(state)    
    else:
        if f_print_log:
            print("--- aff_svc_qna_simple ---")
        state = aff_svc_qna_simple(state)
    
    return state
#init_state = ServiceState(
#    user_question="카리나 고객이 가입한 밀리의서재 제휴사와 관련된 연동 이력을 보여줘",
#    sql_query="",
#    service_type="affiliate",
#    search_results=[],
#    final_answer=""
#)

#aff_svc_qna(init_state)
    

## 3. 채널 연동 서비스 Q&A 구현

### 3-1. 채널 Q&A 데이터 구성

In [83]:
from glob import glob
import warnings
warnings.filterwarnings("ignore")

# LangChain 핵심
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma

# 데이터 처리
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# ollama 임베딩 모델 사용
from langchain_ollama import OllamaEmbeddings

import matplotlib
# 한글 폰트 인식 - Mac
matplotlib.rc('font', family='AppleGothic')
# 마이너스 부호 인식
matplotlib.rc("axes", unicode_minus = False)

# 평가
import ranx_k

# 임베딩 모델을 사용하여 SemanticChunker를 초기화 
#text_splitter = SemanticChunker(
#    embeddings=OllamaEmbeddings(model="bge-m3"),         
#    breakpoint_threshold_type="gradient",  # 임계값 타입 설정 (gradient, percentile, standard_deviation, interquartile)
#)

#ollma 임베딩 사용 여부
embedding_type = "ollama" 

# Chroma 벡터 저장소 생성하기
#chroma_db = Chroma.from_documents(  
#    documents=chunks,
#    embedding=embeddings_ollama,    # 임베딩 사용
#    collection_name="labor_law_rag",    # 컬렉션 이름
#    persist_directory="./chroma_db",
#    collection_metadata = {'hnsw:space': 'cosine'}, # l2, ip, cosine 중에서 선택 
#)

#로그 출력 여부
f_print_log = False

file_patterns = [
    'data/PRJ_CHN_ERR_LISTS.md',
]

def load_text_files(file_patterns):

    documents = []
    
    for pattern in file_patterns:
        files = glob(pattern)
        for file_path in files:
            try:
                loader = TextLoader(file_path, encoding='utf-8')
                docs = loader.load()
                documents.extend(docs)
                if f_print_log:
                    print(f"- load file complete...FILE[{file_path}]")
            except Exception as e:
                if f_print_log:
                    print(f"- !!! load file failed...FILE[{file_path}] ERROR[{e}]")    
    return documents

if f_print_log:
    print("Load file start...")  
raw_documents = load_text_files(file_patterns)
if f_print_log:
    print("-"*80)   

def preprocess_documents(documents):
    """
    문서 전처리 및 메타데이터 추가
    
    Args:
        documents (list): 원본 문서 리스트
    
    Returns:
        list: 전처리된 Document 객체 리스트
    """
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        separators=['\n\n\n'],
        chunk_size=180,
        chunk_overlap=50,
        is_separator_regex=True,
        keep_separator=True,
    )
    chunks = text_splitter.split_documents(documents)
    
    processed_docs = []
    for chunk in chunks:
        # Document 객체 생성
        doc = Document(
            page_content=f"<Document>\n{chunk.page_content}\n</Document>",
            metadata={
                **chunk.metadata,
                'language': 'ko',
                'chunk_length': len(chunk.page_content)
            }
        )
        processed_docs.append(doc)
    
    return processed_docs

processed_docs = preprocess_documents(raw_documents)
if f_print_log:
    print(f"Create chunk complete...CNT[{len(processed_docs)}]")    
    for i, doc in enumerate(processed_docs):
        print(f"\n- C#{i+1} : CHK[{doc.page_content}]")
    print("-"*80)

def load_vector_store(documents, embedding_type: str):
    """
    기존 벡터 저장소를 로드하거나 새로 생성
    
    Returns:
        Chroma: 벡터 저장소 객체
    """

    # 임베딩 선택
    if embedding_type == "ollama":
        embeddings = OllamaEmbeddings(model="bge-m3")
        collection_name="CHN_ERROR_LISTS_OLLAMA"
    else:
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        collection_name="CHN_ERROR_LISTS"

    persist_directory="./chroma_db_tuniverse"
    
    try:
        vector_store = Chroma(
            collection_name=collection_name,
            embedding_function=embeddings,
            persist_directory=persist_directory,
        )
        
        doc_count = vector_store._collection.count()
        if doc_count > 0:
            if f_print_log:
                print(f"Load vector store...CNT[{doc_count}]")
            return vector_store
        else:
            if f_print_log:
                print("!!! Empty vector store...Please add data.")
            
            vector_store = Chroma.from_documents(
                documents=documents,
                embedding=embeddings,
                collection_name=collection_name,
                persist_directory=persist_directory
            )
            
            if f_print_log:
                print(f"Vector store created...CNT[{vector_store._collection.count()}]")
            return vector_store
            
    except Exception as e:
        if f_print_log:
            print(f"!!! Vector store load failed...ERROR[{e}]")
        return None


def initialize_vector_store(embedding_type: str):
    """
    기존 벡터 저장소를 로드하거나 새로 생성
    
    Returns:
        Chroma: 벡터 저장소 객체
    """

    # 임베딩 선택
    if embedding_type == "ollama":
        embeddings = OllamaEmbeddings(model="bge-m3")
        collection_name="CHN_ERROR_LISTS_OLLAMA"
    else:
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        collection_name="CHN_ERROR_LISTS"

    persist_directory="./chroma_db_tuniverse"
    
    try:
        vector_store = Chroma(
            collection_name=collection_name,
            embedding_function=embeddings,
            persist_directory=persist_directory,
        )
        
        doc_count = vector_store._collection.count()
        if doc_count > 0:
            if f_print_log:
                print(f"Load vector store...CNT[{doc_count}]")
            return vector_store
        else:
            if f_print_log:
                print("!!! Empty vector store...Please add data.")                     
            return vector_store
            
    except Exception as e:
        if f_print_log:
            print(f"!!! Vector store load failed...ERROR[{e}]")
        return None

# 벡터 저장소 초기화
#chroma_db = load_vector_store(processed_docs, embedding_type)


### 3-2. 채널 연동 서비스 구현

In [117]:
from langchain_core.tools import tool
from typing import Literal
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents.agent_toolkits import create_retriever_tool

#로그 출력 여부
f_print_log = True

#에이전트 사용 여부
f_use_agent = True

#Retriever 유형 선택
#retriever_type = "S" #similarity
#f_retriever_type = "M" #mmr
#f_retriever_type = "T" #threshold


@tool
def error_reason_search(state: ServiceState):
    """채널 연동 관련 오류 원인을 DB에서 조회하고 결과를 반환하는 도구"""
    # 벡터 저장소 초기화
    chroma_db = initialize_vector_store(state["embedding_type"])

    # 커스텀 검색기 생성
    def create_custom_retriever(search_type="similarity", k=5, **kwargs):
        """커스텀 검색기 생성"""
        return chroma_db.as_retriever(
            search_type=search_type,
            k=k,
            **kwargs
        )
    
    # 다양한 검색기 생성
    if state["retriever_type"] == "S":
        chroma_k_retriever = create_custom_retriever("similarity", k=5)
    elif state["retriever_type"] == "M":
        chroma_k_retriever = create_custom_retriever("mmr", k=5, fetch_k=20)
    elif state["retriever_type"] == "T":
        chroma_k_retriever = create_custom_retriever("similarity_score_threshold", score_threshold=0.8)
    else:
        chroma_k_retriever = create_custom_retriever("similarity", k=5)

    search_db = create_retriever_tool(
        chroma_k_retriever,
        name="search_db",
        description="채널 연동 관련 오류와 해결방안에 대해 데이터베이스에서 검색하고 결과를 반환합니다.",
    )
   
    search_results = search_db.invoke(state["user_question"])
    return search_results

### 3-2. 채널 연동 오류 원인 조사 서비스 구현
def chn_svc_qna_using_agent(state: ServiceState) -> ServiceState:
    """ 채널 연동 관련 오류 원인을 조사하는 함수 """
    
    # 도구 목록
    tools = [error_reason_search]  

    # OpenAI GPT-4.1-mini 모델 사용
    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

    # 도구 실행 에이전트 생성
    search_agent = create_react_agent(llm, tools=tools)

    # 도구 실행 에이전트 사용
    search_results = search_agent.invoke(
        {"messages": [
            ("system", "error_reason_search 도구를 사용"),
            ("human", "{user_question}에 대한 오류 원인을 조사하고 해결방안을 제시해주세요.")
            ]
        }
    )
    state['search_results'] = search_results
    if f_print_log:
        print ("search_results : ", state['search_results'])

    # 결과를 상태에 업데이트 
    return state


### 3-2. 채널 연동 오류 원인 조사 서비스 구현
def chn_svc_qna_simple(state: ServiceState) -> ServiceState:
    """ 채널 연동 관련 오류 원인을 조사하는 함수 """    
   
    # 벡터 저장소 초기화
    chroma_db = initialize_vector_store(state["embedding_type"])

    # 커스텀 검색기 생성
    def create_custom_retriever(search_type="similarity", k=5, **kwargs):
        """커스텀 검색기 생성"""
        return chroma_db.as_retriever(
            search_type=search_type,
            k=k,
            **kwargs
        )

    if state["retriever_type"] == "":
        retriever_type = "S"
    else:
        retriever_type = state["retriever_type"]

    # 다양한 검색기 생성
    if retriever_type == "S":
        chroma_k_retriever = create_custom_retriever("similarity", k=5)
    elif retriever_type == "M":
        chroma_k_retriever = create_custom_retriever("mmr", k=5, fetch_k=20)
    elif retriever_type == "T":
        chroma_k_retriever = create_custom_retriever("similarity_score_threshold", score_threshold=0.8)
    else:
        chroma_k_retriever = create_custom_retriever("similarity", k=5)

    query = state["user_question"]
    search_results = chroma_k_retriever.invoke(query)

    state['search_results'] = search_results
    if f_print_log:
        print ("search_results : ", state['search_results'])

    # 결과를 상태에 업데이트 
    return state

def chn_svc_qna(state: ServiceState) -> ServiceState:
    """ 채널 연동 관련 오류 원인을 조사하는 함수 """
    
    if f_use_agent:
        if f_print_log:
            print("--- chn_svc_qna_using_agent ---")
        state =chn_svc_qna_using_agent(state)    
    else:
        if f_print_log:
            print("--- chn_svc_qna_simple ---")
        state = chn_svc_qna_simple(state)

    # 결과를 상태에 업데이트 
    return state


## 4. 그래프 구성 및 결과 수행

### 4-1. 그래프 구조 설계

In [124]:
from typing import List, TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

# 그래프 구성
builder = StateGraph(ServiceState)

# 노드 추가: 입력 분석, 한국어 문서 검색, 영어 문서 검색, 답변 생성
builder.add_node("chk_svc_typ", chk_svc_typ)
builder.add_node("aff_svc_qna", aff_svc_qna)
builder.add_node("chn_svc_qna", chn_svc_qna)
builder.add_node("revise_qna", revise_qna)
builder.add_node("gen_res", gen_res)

# 엣지 추가: 시작 -> 사용자 입력 분석 -> 조건부 엣지 -> 한국어 문서 검색 또는 영어 문서 검색 -> 답변 생성 -> 끝
builder.add_edge(START, "chk_svc_typ")

# 조건부 엣지 추가
builder.add_conditional_edges(
    "chk_svc_typ",
    decide_next_step
)

builder.add_edge("aff_svc_qna", "gen_res")
builder.add_edge("chn_svc_qna", "gen_res")
builder.add_edge("revise_qna", END)
builder.add_edge("gen_res", END)

# 그래프 컴파일
graph = builder.compile()

# 그래프 시각화
#display(Image(graph.get_graph().draw_mermaid_png()))

### 4-1. 결과 수행

In [138]:
#def call_my_service(self, message: str, history: List) -> str:

f_print_state = False

def call_my_service(message, history):

    history = history or []
    
    init_state = ServiceState(
        user_question=message,
        sql_query="",
        service_type="",
        need_to_revise=False,
        retriever_type="S", # S : similarity, M : mmr, T : threshold
        embedding_type="ollama", # ollama, openai
        search_results=[],
        final_answer=""
    )

        # 그래프 실행
    result = graph.invoke(init_state)

    show_state(result)

    response = result['final_answer']

    if f_print_state:
        response = merge_state(result)
    
    history.append((message, response))
    
    # 결과 출력
    #return result['final_answer']
    return history, history


qust = "카리나 고객님 010410892777 스타벅스 상품 관련"
hist = []
call_my_service(qust,hist)

=============== chk_svc_typ ===============
Use lanfuse!!! chk_svc_type
Use lanfuse!!! #0 : 
Use lanfuse!!! #1 :  input_variables=['user_question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['user_question'], input_types={}, partial_variables={}, template='\n당신은 사용자 입력을 통해 질문 영역을 판단하고 정해진 답변을 제공하는 전문가입니다.\n\n질문 영역은 "제휴사" 혹은 "채널" 영역이 존재합니다.\n다음 제시된 조건에 하나라도 완전히 부합하는 경우 질문 영역을 판단할 수 있습니다.\n제시된 조건은 각각 독립적이며 완전히 부합하는 조건이 하나도 없는 경우 질문 영역을 판단할 수 없고 이 경우 "unknown"으로 답변합니다.\n제시된 조건은 우선순위의 내림차순이므로 상위 조건 중 완전히 부합하는 조건이 발견되면 하위 조건은 검사하지 않고 영역을 판단하여 답변합니다.\n[조건 1-1] 오류코드(400,401,403,404,500과 같은 399보다 크고 600보다 작은 3자리 숫자)가 존재하는 경우 "채널" 영역으로 판단\n[조건 1-2] 오류메시지(Access Restricted, Forbidden, Unauthorized, Bad Request, Not Found, Internal Server Error)가 존재하는 경우 "채널" 영역으로 판단\n[조건 2-1] 계약번호(3으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 영역으로 판단 \n[조건 2-2] 계약서비스번호(6으로 시작하는 10자리숫자)가 존재하는 경우 "제휴사" 영역으로 판단 \n[조건 2-3] 고객(회원)이름(한글2자이상)과 이동전화번호(010으로 시작하는 13

Use lanfuse!!! #3 :  content='"제휴사" 영역 판단 조건 중\n- 고객 이름(한글 2자 이상): "카리나" (한글 2자 이상 아님, 한글+영문 혼용)\n- 이동전화번호(010으로 시작하는 13자리 숫자): "010410892777" (11자리, 13자리 아님)\n- 상품명(3자리 이상 문자): "스타벅스" (충분)\n\n고객 이름은 한글 2자 이상이어야 하는데 "카리나"는 한글+영문 혼용이라 부합하지 않고, 이동전화번호도 13자리가 아님(총 11자리).\n\n따라서 조건 2-3은 완전히 부합하지 않음.\n\n다른 조건도 만족하지 않으므로 판단 불가.\n\n답변: unknown' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 177, 'prompt_tokens': 516, 'total_tokens': 693, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4fce0778af', 'id': 'chatcmpl-CLmo4IScbJGajI0s12cS4KEz5rfk5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--dabecdeb-d4b5-46db-8fde-63e1407e5374-0' usage_metadata={'input_tokens': 516, 'output_tokens': 177, 'total_t

AttributeError: 'AIMessage' object has no attribute 'strip'

In [112]:
import gradio as gr

with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="slate",
        secondary_hue="blue",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Noto Sans")]
    )
) as demo:
    gr.HTML(
        """
        <style>
            .gradio-container {
                    background: url("https://raw.githubusercontent.com/LeoLee-likedawn/llm-foundation-lab-project/main/images/PRJ-%E1%84%87%E1%85%A2%E1%84%80%E1%85%A7%E1%86%BC%E1%84%92%E1%85%AA%E1%84%86%E1%85%A7%E1%86%AB-1001.png") 
                                no-repeat center center fixed !important;
                    background-size: cover !important;
            }
        </style>
        """
    )
    gr.Markdown(
        """
        <div style='display:flex; align-items:center; justify-content:center; gap:10px;'>
            <img src="https://github.com/LeoLee-likedawn/llm-foundation-lab-project/raw/main/images/tuniverse.png" width="40" height="40" style="border-radius:50%;" />
            <h1 style='margin:0; font-size:24px;'>T우주 연동 Q&A 서비스</h1>
        </div>
        <p style='text-align:center; font-size:16px; color:gray;'>

        
        </p>
        """,
    )

    chatbot_ui = gr.Chatbot(
        type="messages",
        label="대화창",
        avatar_images=("images/user.png", "images/tbot.png"),  # 사용자/봇 아바타
        height=500,
    )

    user_input = gr.Textbox(
        placeholder="메시지를 입력하세요...",
        show_label=False,
        container=False,
    )
    submit_btn = gr.Button("보내기", variant="primary")

    #state = gr.State([])

    #submit_btn.click(call_my_service, [user_input, state], [chatbot_ui, state])
    #user_input.submit(call_my_service, [user_input, state], [chatbot_ui, state])

    #chatbot_ui = gr.Chatbot(elem_classes="big-chatbot")
    #msg = gr.Textbox(placeholder="질문을 입력하세요...")
    clear = gr.Button("Clear")

    user_input.submit(call_my_service, [user_input, chatbot_ui], [chatbot_ui, chatbot_ui])
    user_input.submit(lambda: "", None, user_input)
    clear.click(lambda: None, None, chatbot_ui)

demo.launch()

* Running on local URL:  http://127.0.0.1:7901
* To create a public link, set `share=True` in `launch()`.


In [45]:
import gradio as gr

# 간단한 챗봇 함수
#def chatbot(message, history):
#    history = history or []
#    response = f"봇: {message}"
#    history.append((message, response))
#    return history, history

#with gr.Blocks(title="T우주 연동 Q&A") as demo:
with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="slate",
        secondary_hue="blue",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Noto Sans")]
    )
) as demo:
    # CSS 삽입 (파스텔톤 하늘색 배경)
    gr.HTML(
        """
        <style>
            body {
                background-color: #aee6f9; /* 파스텔톤 하늘색 배경 */
            }
            .big-chatbot {
                height: 300px !important;
                width: 100% !important;
                font-size: 12px;
            }
            /* 사용자 메시지 */
            .big-chatbot [data-role="user"] .message {
                background-color: #d7f3f7 !important;  /* 파스텔 하늘색 */
                color: #000 !important;
                font-size: 12px !important;
                border-radius: 12px;
                padding: 8px 12px;
            }
            /* 봇 메시지 */
            .big-chatbot [data-role="assistant"] .message {
                background-color: #e6d7f7 !important;  /* 파스텔 보라색 */
                color: #000 !important;
                font-size: 12px !important;
                border-radius: 12px;
                padding: 8px 12px;
            }
        </style>
        """
    )

    gr.Markdown(
        """
        <div style='display:flex; align-items:center; justify-content:center; gap:10px;'>
            <img src="https://github.com/LeoLee-likedawn/llm-foundation-lab-project/raw/main/images/tuniverse.png" width="40" height="40" style="border-radius:50%;" />
            <h1 style='margin:0; font-size:24px;'>T우주 연동 Q&A 서비스</h1>
        </div>
        <p style='text-align:center; font-size:16px; color:gray;'>
        
        </p>
        """
    )

    #chatbot_ui = gr.Chatbot(elem_classes="big-chatbot")
    chatbot_ui = gr.Chatbot(
        elem_classes="big-chatbot",
        label="대화창",
        avatar_images=("images/user.png", "images/tbot.png"),  # 사용자/봇 아바타
        height=500,
    )
    
    msg = gr.Textbox(
        placeholder="문의사항을 입력하세요...",
        show_label=False,
        container=False,
    )
    clear = gr.Button("Clear")

    msg.submit(call_my_service, [msg, chatbot_ui], [chatbot_ui, chatbot_ui])
    msg.submit(lambda: "", None, msg)
    clear.click(lambda: None, None, chatbot_ui)

demo.launch()

* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.


---------------------------------  State Info.   ---------------------------------
- state.service_type :  channel
- state.user_question :  500 오류 원인과 예시
- state.retriever_type :  S
- state.embedding_type :  ollama
- state.sql_query :  
- state.search_results :  [Document(id='d278df88-4aff-406f-a705-e149ce2763cb', metadata={'language': 'ko', 'chunk_length': 380, 'source': 'data/PRJ_CHN_ERR_LISTS.md'}, page_content='<Document>\n\n\n\n\n### CASE08 - 500/Internal Server Error\n## http 상태 코드\n500\n## http 상태 메시지\nInternal Server Error\n## http 응답 바디\n{\n    "timestamp": ""20241205175222",\n    "status"": 500,\n    "code"": "COM_9999"",\n    "message"": "[COM_9999] 내부 오류가 발생하였습니다"\n}\n## 발생원인\n전문 구성 오류\n## 해결방안\n요청 바디의 구성에 오류가 있거나 바디의 구성이 사전 정의된 것과 다르게 구성된 경우 발생 가능\n요청 전문의 구조(JSON)가 정상인지 확인\n이중따움표 누락이나 잘못된 위치에 배치하는 경우 빈번하게 발생\n</Document>'), Document(id='9304dbce-68c7-404a-ba24-9539f66a2f32', metadata={'chunk_length': 310, 'source': 'data/PRJ_CHN_ERR_LISTS.md', 'language': 'ko'}, page_conte

In [ ]:
import gradio as gr

with gr.Blocks(title="T우주 연동 Q&A") as demo:
        # CSS 삽입 (파스텔톤 하늘색 배경)
        gr.HTML(
            """
            <style>
                body {
                    background-color: #aee6f9; /* 파스텔톤 하늘색 배경 */
                }
                .big-chatbot {
                    height: 800px !important;
                    width: 100% !important;
                    font-size: 12px;
                }
                /* 사용자 메시지 (홀수 줄) */
                .big-chatbot .wrap:nth-child(odd) .message {
                    font-size: 8px !important; /* 원하는 크기로 변경 */
                    background-color: #d7f3f7 !important;  /* 파스텔 하늘색 */
                    color: #000;
                    border-radius: 12px;
                    padding: 8px 12px;
                }
                /* 봇 메시지 (짝수 줄) */
                .big-chatbot .wrap:nth-child(even) .message {
                    font-size: 8px !important; /* 원하는 크기로 변경 */
                    background-color: #e6d7f7 !important;  /* 파스텔 보라색 */
                    color: #000;
                    border-radius: 12px;
                    padding: 8px 12px;
                }
            </style>
            """
        )

        gr.Markdown("# T우주 연동 Q&A")

        chatbot_ui = gr.Chatbot(elem_classes="big-chatbot")
        msg = gr.Textbox(placeholder="질문을 입력하세요...")
        clear = gr.Button("Clear")

        msg.submit(call_my_service, [msg, chatbot_ui], [chatbot_ui, chatbot_ui])
        msg.submit(lambda: "", None, msg)
        clear.click(lambda: None, None, chatbot_ui)

demo.launch()